# **OPTIONAL TUTORIAL**: Calculate allocated carrying capacities with pyaesa

Absolute environmental sustainability assessment compares environmental burdens
with carrying capacities allocated to the assessed activity. This notebook uses
[pyaesa](https://github.com/AESAtoolkit/pyaesa) 1.3.0 to calculate allocated
carrying capacities for the plastics sector in each region and write the inputs
used by pyToCE for monitoring and target setting.

## Before starting...

Complete [Tutorial 4](../core_workflow/4_run_lca.ipynb) and locate its 2019 EF 3.1
plastics LCA results. These burdens are used for acquired rights allocation.

Run this notebook in a separate Python 3.11, 3.12, 3.13, or 3.14 environment.
pyToCE uses PyArrow 24, while pyaesa 1.3.0 requires PyArrow below version 24.
Install the notebook environment with:

```console
python -m pip install "pyaesa==1.3.0" "PyYAML==6.0.3" "jupyterlab>=4,<5"
```

pyaesa runs in its own workspace. The final YAML and CSV files are written to
the pyToCE workspace selected below.


## Choose the workspaces and output statistics

Edit `PYTOCE_WORKSPACE` and `PYAESA_WORKSPACE` before running the notebook.
Set `TUTORIAL_DATA` to the supplied
`tutorials/optional_workflows/data/pyaesa_plastics/` directory. It contains the
two CSV files described in the [aggregation guide](data/pyaesa_plastics/README.md).
Replace `<lca_result>` in `CE_LCA_2019_RESULTS` with the result directory from
Tutorial 4.

The output contains annual allocated carrying capacities for all 16 EF 3.1
impact categories over 2019--2100 and cumulative GWP100 allocated carrying
capacities for 2019--2100. `ALLOCATED_CC_RESULT="pyaesa_plastics"` names the
directory written under `data_raw/allocated_cc/results/` in the pyToCE
workspace. Tutorials 5m and 5o use this name.

By default, `SELECTED_STATISTICS` contains `p5`, `p50`, and `p95`: the 5th,
50th (median), and 95th percentiles of the allocated carrying capacity
distributions. Each percentile is used as a separate deterministic input to
the ASR calculation. The other supported statistics are `mean`, `min`, `max`,
`p25`, and `p75`.

With `INTER_METHOD_UNCERTAINTY=True`, sharing principles receive equal weights
and their applicable enacting metrics receive equal conditional weights.
Allocation paths are combined through Monte Carlo sampling, and each output case
is named by its statistic. With `False`, each allocation method is retained
separately, with labels of the form `<allocation_method>_<statistic>` in lowercase
`snake_case`. Select these labels through pyToCE's `allocated_cc_case` argument.

The data download steps require network access. `pyaesa` requires at least
**4 GB of available RAM** to run.


In [ ]:
import shutil
from pathlib import Path

import pandas as pd
import yaml
from pyaesa import (
    download_ar6,
    download_mrio,
    download_pop_gdp,
    prepare_external_inputs,
    process_mrio,
    process_pop_gdp,
    uncertainty_acc,
)
from pyaesa import (
    set_workspace as set_pyaesa_workspace,
)

PYTOCE_WORKSPACE = Path.home() / "Documents" / "pytoce_workspace"
PYAESA_WORKSPACE = Path.home() / "Documents" / "UNCASExt_demo"

TUTORIAL_DATA = Path(
    "path/to/tutorials/optional_workflows/data/pyaesa_plastics"
).resolve()
CE_LCA_2019_RESULTS = (
    PYTOCE_WORKSPACE
    / "results/frozen_2019_default_baseline/lca_results/<lca_result>/annual_scores.parquet"
)

INTER_METHOD_UNCERTAINTY = True
SELECTED_STATISTICS = ("p5", "p50", "p95")
ALLOCATED_CC_RESULT = "pyaesa_plastics"

set_pyaesa_workspace(PYAESA_WORKSPACE)

## Add the plastics aggregation files

The tutorial supplies two mappings:

- [agg_reg_h12_plast.csv](data/pyaesa_plastics/agg_reg_h12_plast.csv) maps
  EXIOBASE to the 12 regions used by pyToCE.
- [agg_sec_h12_plast.csv](data/pyaesa_plastics/agg_sec_h12_plast.csv) aggregates
  basic plastics production, secondary plastics reprocessing, plastic waste
  incineration, and plastic waste landfill into `Plastics`.

EXIOBASE countries are assigned to their corresponding pyToCE region.
Rest-of-World regions are distributed using annual GDP shares. The
[aggregation guide](data/pyaesa_plastics/README.md) describes the mappings,
data sources, and citation.

The next cell copies the files into the pyaesa workspace before the MRIO steps:

- `{PYAESA_WORKSPACE}/data_raw/mrio/exiobase_3/aggregation/agg_reg_h12_plast.csv`
- `{PYAESA_WORKSPACE}/data_raw/mrio/exiobase_3/aggregation/pxp/agg_sec_h12_plast.csv`


In [ ]:
SOURCE = "exiobase_3102_pxp"
AGGREGATION_VERSION = "h12_plast"
FUNCTIONAL_UNIT = "L2.a.b"
SECTOR = "Plastics"
MRIO_YEARS = range(1995, 2025)
ALLOCATED_CC_YEARS = range(2019, 2101)

REG_AGGREGATION_SOURCE = TUTORIAL_DATA / 'agg_reg_h12_plast.csv'
REG_AGGREGATION_TARGET = PYAESA_WORKSPACE / 'data_raw/mrio/exiobase_3/aggregation/agg_reg_h12_plast.csv'
SEC_AGGREGATION_SOURCE = TUTORIAL_DATA / 'agg_sec_h12_plast.csv'
SEC_AGGREGATION_TARGET = PYAESA_WORKSPACE / 'data_raw/mrio/exiobase_3/aggregation/pxp/agg_sec_h12_plast.csv'

REG_AGGREGATION_TARGET.parent.mkdir(parents=True, exist_ok=True)
SEC_AGGREGATION_TARGET.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(REG_AGGREGATION_SOURCE, REG_AGGREGATION_TARGET)
shutil.copy2(SEC_AGGREGATION_SOURCE, SEC_AGGREGATION_TARGET)
print("Copied regional and sector aggregation files into pyaesa workspace.")
print(f"  {REG_AGGREGATION_SOURCE.name} -> {REG_AGGREGATION_TARGET}")
print(f"  {SEC_AGGREGATION_SOURCE.name} -> {SEC_AGGREGATION_TARGET}")


## Choose the allocation methods

The assessed activity includes plastics production and end-of-life activities
occurring in each region, together with their upstream supply chains, serving
total demand worldwide. This corresponds to the `L2.a.b` functional unit.

The example uses the eight allocation paths described in the Methodological
appendix: the one-step utilitarian `UT(TD)` path, the one-step acquired rights
`AR(E^{CBA_TD})` path, and six two-step paths. Each two-step path first allocates
a country-level share using population, GDP per capita, or historical
responsibility, then allocates a share to the plastics sector using `UT(FDa)`
or `UT(GVAa)`.

The next cell defines the seven native pyaesa paths. The acquired rights shares
are calculated from the plastics LCA and supplied below as external inputs.
Definitions and method identifiers are given in pyaesa's
[Functional Units and Allocation Methods guide for version 1.3.0](https://github.com/AESAtoolkit/pyaesa/blob/v1.3.0/tutorials/study_objectives/1_functional_units_and_allocation_methods.md).


In [ ]:
ef31_impact_codes = {
    "acidification": "AC",
    "climate_change": "GWP_100",
    "ecotoxicity_freshwater": "ECOTOX",
    "energy_resources_non_renewable": "FRD",
    "eutrophication_freshwater": "FEU",
    "eutrophication_marine": "MEU",
    "eutrophication_terrestrial": "TEU",
    "human_toxicity_carcinogenic": "HTOX_c",
    "human_toxicity_non_carcinogenic": "HTOX_nc",
    "ionising_radiation_human_health": "IR",
    "land_use": "LU",
    "material_resources_metals_minerals": "MRD",
    "ozone_depletion": "ODP",
    "particulate_matter_formation": "PM",
    "photochemical_oxidant_formation_human_health": "POF",
    "water_use": "WU",
}
native_method_plan = {
    "method_plan": "one_step_pairs",
    "one_step_methods": ["UT(TD)"],
    "l1_l2_pairs": [
        "EG(Pop)::UT(FDa)",
        "PR(GDPcap)::UT(FDa)",
        "PR-HR(Ecap,cum^{CBA_FD})::UT(FDa)",
        "EG(Pop)::UT(GVAa)",
        "PR(GDPcap)::UT(GVAa)",
        "PR-HR(Ecap,cum^{PBA})::UT(GVAa)",
    ],
    "ssp_scenario": ["SSP2"],
}


### Allocation LCIA inputs

Historical responsibility uses cumulative environmental pressures over the
configured responsibility period. As described in the Methodological appendix,
this example uses cumulative GWP100 as the historical-responsibility metric for
all 16 EF 3.1 impact categories. Acquired rights instead uses the plastics LCA
results for each EF 3.1 category.

An [EF 3.1 concordance matrix for EXIOBASE 3.10.2](https://doi.org/10.5281/zenodo.21533474)
is available. The Methodological appendix reports differences between the global
burdens calculated with this matrix and the EF 2010 global normalization factors,
associated with environmental extensions and elementary-flow mapping. It
therefore uses process-LCA-based acquired rights shares and GWP100-based
historical responsibility, as implemented here.

The following cell replicates the GWP100 characterization matrix under each EF
category label and copies its configured responsibility period. The installed
`gwp100_lcia_rps.csv` uses a 300-year responsibility period; GWP100 uses a
100-year characterization horizon. pyaesa calculates historical responsibility
from the available LCIA history within the configured period, ending at the
latest available LCIA year at or before the assessment year. This example
processes 1995--2024.

The cell also copies the EF 3.1 steady-state carrying capacity file under the
custom LCIA name used for this allocation calculation.


In [ ]:
lcia_root = PYAESA_WORKSPACE / "data_raw/mrio/exiobase_3/lcia"
gwp100 = pd.read_csv(lcia_root / "characterization_factors_matrices/gwp100_lcia.csv")
ef31_cc_path = PYAESA_WORKSPACE / "data_raw/carrying_capacities/ef_3.1_cc_steady_state.csv"
ef31_cc = pd.read_csv(ef31_cc_path)
custom_lcia = pd.concat(
    [
        gwp100.assign(
            impact_full_name=category.impact_full_name,
            impact_parent=category.impact,
            impact=category.impact,
            impact_unit=category.impact_unit,
        )
        for category in ef31_cc.itertuples(index=False)
    ],
    ignore_index=True,
)
custom_lcia.to_csv(
    lcia_root / "characterization_factors_matrices/ef31_gwp100_allocation.csv",
    index=False,
)
gwp100_periods = pd.read_csv(lcia_root / "responsibility_periods/gwp100_lcia_rps.csv")
custom_periods = pd.concat(
    [
        gwp100_periods.assign(
            impact_full_name=category.impact_full_name,
            impact_parent=category.impact,
            impact=category.impact,
        )
        for category in ef31_cc.itertuples(index=False)
    ],
    ignore_index=True,
)
custom_periods.to_csv(
    lcia_root / "responsibility_periods/ef31_gwp100_allocation_rps.csv",
    index=False,
)
static_cc_path = (
    PYAESA_WORKSPACE
    / "data_raw/carrying_capacities/ef31_gwp100_allocation_cc_steady_state.csv"
)
shutil.copy2(ef31_cc_path, static_cc_path)

### Calculate shares using acquired rights and the plastics LCA

For each impact category and region, the 2019 environmental burden of the
plastics sector is divided by the corresponding EF 3.1 2010 global
normalization factor. The resulting allocated shares of carrying capacities
(aSoCC) are supplied under the external method label `AR(E^{CBA_TD})` and
kept constant from 2019 to 2100.

The allocation reference year is 2019, while the global normalization factors
describe environmental pressures in 2010. The next cells call
`prepare_external_inputs(...)` to create pyaesa's input directories and write
these shares to CSV files. pyaesa reads them alongside the selected native
allocation methods.


In [ ]:
ce_scores = pd.read_parquet(CE_LCA_2019_RESULTS, filters=[("year", "==", 2019)])
ce_scores = ce_scores.loc[
    ce_scores["method_id"].eq("ef31")
    & ce_scores["region"].ne("global_derived")
    & ce_scores["impact_category_id"].isin(ef31_impact_codes)
].copy()
ce_scores["impact"] = ce_scores["impact_category_id"].map(ef31_impact_codes)
ce_scores = ce_scores.rename(columns={"region": "r_p", "score": "lca_value"})
normalization = pd.read_csv(static_cc_path)[
    ["impact", "global_normalisation_factor"]
]
acquired_rights = ce_scores[["r_p", "impact", "lca_value"]].merge(
    normalization, on="impact", how="left", validate="many_to_one"
)
acquired_rights["allocated_share"] = (
    acquired_rights["lca_value"]
    / acquired_rights["global_normalisation_factor"]
)
acquired_rights["s_p"] = SECTOR
acquired_rights["reference_year"] = 2019

#### Add the acquired rights shares to the pyaesa workspace

Write the acquired rights shares for each assessment year in both pyaesa
projects: `plastics_ef31` for annual EF 3.1 allocated carrying capacities and
`plastics_gwp100` for cumulative climate change allocated carrying capacities.
The files supply `AR(E^{CBA_TD})` shares with reference year 2019.


In [ ]:
for project_name, lcia_method, selected in (
    (
        "plastics_gwp100",
        "gwp100_lcia",
        acquired_rights.loc[acquired_rights["impact"].eq("GWP_100")],
    ),
    ("plastics_ef31", "ef31_gwp100_allocation", acquired_rights),
):
    external_table = selected[
        ["r_p", "s_p", "impact", "reference_year"]
    ].reset_index(drop=True)
    external_table = pd.concat(
        [
            external_table,
            pd.DataFrame(
                {
                    str(year): selected["allocated_share"].to_numpy()
                    for year in ALLOCATED_CC_YEARS
                }
            ),
        ],
        axis=1,
    )
    external_inputs = prepare_external_inputs(project_name=project_name)
    external_table.to_csv(
        external_inputs.external_asocc_root
        / "deterministic"
        / f"AR(E^{{CBA_TD}})__{lcia_method}.csv",
        index=False,
        float_format="%.17g",
    )
external_method = {"one_step_methods": ["AR(E^{CBA_TD})"]}

## Download and process the pyaesa data

These functions write to the selected pyaesa workspace. Their outputs provide
the population, GDP, economic, and environmental data used by the allocation
methods, along with the pathways used to define dynamic climate change carrying
capacities:

- Historical population and GDP from the World Bank, with SSP2 values used for
  prospective years. pyaesa uses a historical year column when available and the
  selected SSP2 values otherwise.
- EXIOBASE 3.10.2 tables from 1995 through 2024, including the nowcasted 2023 and
  2024 tables.
- IPCC AR6 GHG emissions pathways for dynamic climate change carrying capacities.

The prospective allocation methods use these data differently. `UT(TD)`
projects regional plastics sector output and global final demand from
EXIOBASE 1995--2022 and SSP2 GDP using the UNCASExt regression approach.
`UT(FDa)` and `UT(GVAa)` instead reuse historical MRIO structures. In each
Monte Carlo draw, one structure from 1995--2022 is sampled with equal probability
and used for prospective allocation.

For historical responsibility, cumulative per-capita environmental pressures
are calculated through the latest available LCIA year at or before the assessment
year. With the LCIA history processed here, the historical-responsibility
priority factor is fixed after 2024, while its population-weighting component
continues to follow SSP2.

The example uses fixed LCIA inputs and one MRIO source. Acquired rights uses
the fixed 2019 reference year selected above.


In [ ]:
download_pop_gdp()
process_pop_gdp()
download_ar6()
download_mrio(SOURCE, years=MRIO_YEARS)
process_mrio(
    SOURCE,
    years=MRIO_YEARS,
    # Historical responsibility uses cumulative GWP100 across all EF 3.1 categories.
    lcia_method=("gwp100_lcia", "ef31_gwp100_allocation"),
    agg_reg=True,
    agg_sec=True,
    agg_version=AGGREGATION_VERSION,
)

## Calculate annual and cumulative allocated carrying capacities

Allocated carrying capacities are obtained by multiplying allocated shares of
carrying capacities by the corresponding global carrying capacities. The two
`uncertainty_acc(...)` calls below create or reuse the required allocated
shares and dynamic climate change carrying capacity results, then propagate the
selected uncertainty sources through Monte Carlo sampling.

The first call uses the dynamic climate change carrying capacity derived from
SSP2 pathways in
[IPCC AR6 categories C1--C4](https://www.ipcc.ch/report/ar6/wg3/chapter/chapter-3/).
In each Monte Carlo draw, one category is sampled with equal probability,
followed by one eligible SSP2 model-scenario pathway using simple random
sampling. The carrying capacity uses GWP100 Kyoto-gas emissions excluding
agriculture, forestry, and other land use (AFOLU), with pyaesa's `gross_alt`
emissions definition. Annual allocated values are summed within each draw over
2019--2100.

The second call calculates annual allocated carrying capacities for the 16 EF
3.1 impact categories using steady-state carrying capacities. Both calls use
pyaesa's default Monte Carlo convergence mode and produce summary statistics and
figures. The following cells read the annual and cumulative summaries to prepare
the pyToCE input.


In [ ]:
uncertainty_config = {
    "inter_method_uncertainty": {
        "active": INTER_METHOD_UNCERTAINTY,
    },
}

dynamic_report = uncertainty_acc(
    project_name="plastics_gwp100",
    source=SOURCE,
    agg_reg=True,
    agg_sec=True,
    agg_version=AGGREGATION_VERSION,
    years=ALLOCATED_CC_YEARS,
    fu_code=FUNCTIONAL_UNIT,
    s_p=SECTOR,
    lcia_method="gwp100_lcia",
    base_asocc_args=native_method_plan,
    external_method=external_method,
    base_cc_args={
        "static": {"active": False},
        "dynamic_ar6": {
            "active": True,
            "ssp_scenario": ["SSP2"],
            # pyaesa defaults: emissions_mode="gross_alt", include_afolu=False.
        },
    },
    # Sample C1-C4 with equal probability, then use simple random sampling
    # among eligible SSP2 pathways within the selected category.
    uncertainty_config={
        **uncertainty_config,
        "dynamic_ar6_cc_uncertainty": {"category_uncertainty": True},
    },
    output_format="parquet",
    figures=True,
    subfigures=False,
)

static_report = uncertainty_acc(
    project_name="plastics_ef31",
    source=SOURCE,
    agg_reg=True,
    agg_sec=True,
    agg_version=AGGREGATION_VERSION,
    years=ALLOCATED_CC_YEARS,
    fu_code=FUNCTIONAL_UNIT,
    s_p=SECTOR,
    lcia_method="ef31_gwp100_allocation",
    base_asocc_args=native_method_plan,
    external_method=external_method,
    base_cc_args={
        "static": {"active": True, "exclude_max_cc": True},
        "dynamic_ar6": {"active": False},
    },
    uncertainty_config=uncertainty_config,
    output_format="parquet",
    figures=True,
    subfigures=False,
)

## Write the pyToCE input files

Select the requested statistics from the annual and cumulative pyaesa summaries
and write the YAML definition and two CSV files required by pyToCE. With
inter-method uncertainty disabled, each case label combines its allocation
method and statistic.


In [ ]:
statistic_columns = {
    "mean": "mean",
    "min": "min",
    "max": "max",
    "p5": "p5",
    "p25": "p25",
    "p50": "median",
    "p75": "p75",
    "p95": "p95",
}
selected_columns = {
    statistic_columns[statistic]: statistic for statistic in SELECTED_STATISTICS
}
scope = "inter_method" if INTER_METHOD_UNCERTAINTY else "per_method"
impact_ids = {value: key for key, value in ef31_impact_codes.items()}
unit_labels = {
    ("AC", "mol H+ eq"): "mol H+-Eq",
    ("GWP_100", "kg CO2-eq"): "kg CO2-Eq",
    ("ECOTOX", "CTUe"): "CTUe",
    ("FRD", "MJ"): "MJ, net calorific value",
    ("FEU", "kg P-eq"): "kg P-Eq",
    ("MEU", "kg N-eq"): "kg N-Eq",
    ("TEU", "mol N-eq"): "mol N-Eq",
    ("HTOX_c", "CTUh"): "CTUh",
    ("HTOX_nc", "CTUh"): "CTUh",
    ("IR", "kBq U-235-eq"): "kBq U235-Eq",
    ("LU", "Pt"): "dimensionless",
    ("MRD", "kg Sb-eq"): "kg Sb-Eq",
    ("ODP", "kg CFC-11-eq"): "kg CFC-11-Eq",
    ("PM", "Disease incidence"): "disease incidence",
    ("POF", "kg NMVOC-eq"): "kg NMVOC-Eq",
    ("WU", "m3 water-eq"): "m3 world Eq deprived",
}

### Annual allocated carrying capacities

The next cell selects the requested statistics for annual allocated carrying
capacities. It maps the impact category and unit labels to those used by pyToCE,
retaining the calculated values and their source record.


In [ ]:
static_summary_path = Path(static_report.manifest.artifacts["summary_stats_runs"])
annual_source = pd.read_parquet(static_summary_path)
annual_source = annual_source.loc[
    annual_source["acc_summary_scope"].eq(scope)
    & annual_source["cc_type"].eq("static")
    & annual_source["cc_bound"].eq("min_cc")
    & annual_source["year"].isin(ALLOCATED_CC_YEARS)
    & annual_source["impact"].isin(impact_ids)
].copy()
annual_source = annual_source.rename(columns=selected_columns)

if INTER_METHOD_UNCERTAINTY:
    annual_source["allocation_selection"] = "inter_method_equal_weight"
else:
    annual_source["allocation_selection"] = annual_source[
        ["l1_l2_method", "l2_method", "l1_method"]
    ].bfill(axis=1).iloc[:, 0]
annual = annual_source.melt(
    id_vars=["r_p", "impact", "impact_unit", "year", "allocation_selection"],
    value_vars=list(SELECTED_STATISTICS),
    var_name="statistic",
    value_name="allocated_cc",
)
method_tokens = (
    annual["allocation_selection"]
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
annual["allocated_cc_case"] = (
    annual["statistic"]
    if INTER_METHOD_UNCERTAINTY
    else method_tokens + "_" + annual["statistic"]
)
annual["impact_category_id"] = annual["impact"].map(impact_ids)
annual["lcia_method"] = "ef31"
annual["region"] = annual["r_p"]
annual["unit"] = [
    unit_labels[(impact, unit)]
    for impact, unit in zip(annual["impact"], annual["impact_unit"], strict=True)
]
annual["cc_authority"] = "pyaesa=1.3.0; source=annual_static_min_cc"
annual = annual[
    [
        "allocated_cc_case",
        "lcia_method",
        "region",
        "impact_category_id",
        "year",
        "allocated_cc",
        "unit",
        "cc_authority",
    ]
].sort_values(
    ["allocated_cc_case", "lcia_method", "region", "impact_category_id", "year"],
    kind="stable",
    ignore_index=True,
)

### Cumulative allocated dynamic climate change carrying capacity

Read the cumulative 2019--2100 GWP100 statistics for SSP2. For each Monte Carlo
draw, pyaesa first multiplies the annual allocated share by the dynamic climate
change carrying capacity and sums these products over 2019--2100. The requested
percentiles are then calculated from the resulting cumulative distribution.


In [ ]:
cumulative_summary_path = Path(
    dynamic_report.manifest.artifacts["cumulative_summary_stats_runs"]
)
cumulative_source = pd.read_parquet(cumulative_summary_path)
cumulative_source = cumulative_source.loc[
    cumulative_source["acc_summary_scope"].eq(scope)
    & cumulative_source["cc_type"].eq("dynamic_ar6")
    & cumulative_source["cc_flow"].eq("positive_emissions")
    & cumulative_source["impact"].eq("GWP_100")
    & cumulative_source["ar6_cc_ssp_scenario"].eq("SSP2")
].copy()
cumulative_source = cumulative_source.rename(columns=selected_columns)

if INTER_METHOD_UNCERTAINTY:
    cumulative_source["allocation_selection"] = "inter_method_equal_weight"
else:
    cumulative_source = cumulative_source.loc[
        cumulative_source["asocc_ssp_scenario"].eq("SSP2")
        | cumulative_source["reference_year"].notna()
    ]
    cumulative_source["allocation_selection"] = cumulative_source[
        ["l1_l2_method", "l2_method", "l1_method"]
    ].bfill(axis=1).iloc[:, 0]
cumulative = cumulative_source.melt(
    id_vars=["r_p", "impact", "impact_unit", "allocation_selection"],
    value_vars=list(SELECTED_STATISTICS),
    var_name="statistic",
    value_name="cumulative_allocated_cc",
)
method_tokens = (
    cumulative["allocation_selection"]
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
cumulative["allocated_cc_case"] = (
    cumulative["statistic"]
    if INTER_METHOD_UNCERTAINTY
    else method_tokens + "_" + cumulative["statistic"]
)
cumulative["region"] = cumulative["r_p"]
cumulative["impact_category_id"] = "climate_change"
cumulative["period_start"] = 2019
cumulative["period_end"] = 2100
cumulative["unit"] = [
    unit_labels[(impact, unit)]
    for impact, unit in zip(
        cumulative["impact"], cumulative["impact_unit"], strict=True
    )
]
cumulative["budget_metric"] = "ef31_climate_change_gwp100_all_ghg"
cumulative["cc_authority"] = "pyaesa=1.3.0; source=dynamic_ar6_ssp2"
cumulative = cumulative[
    [
        "allocated_cc_case",
        "region",
        "impact_category_id",
        "period_start",
        "period_end",
        "cumulative_allocated_cc",
        "unit",
        "budget_metric",
        "cc_authority",
    ]
].sort_values(["allocated_cc_case", "region"], kind="stable", ignore_index=True)

### Write the pyToCE result directory

Write `allocated_cc_result.yaml`, `annual_allocated_cc.csv`, and
`cumulative_gwp100_allocated_cc.csv` to the pyToCE workspace. Choose a new
`ALLOCATED_CC_RESULT` name when writing revised allocated carrying capacities.

After writing these files, return to the pyToCE environment and choose your
study objective:

- **Monitoring:** continue with
  [Tutorial 5m](../core_workflow/5m_run_asr.ipynb) to assess user-defined CE
  performance trajectories.
- **Target setting:** continue with
  [Tutorial 5o](../core_workflow/5o_run_optimization.ipynb) to search for CE
  performance target levels.

Both notebooks use the directory name set by `ALLOCATED_CC_RESULT`.


In [ ]:
result_directory = (
    PYTOCE_WORKSPACE / "data_raw/allocated_cc/results" / ALLOCATED_CC_RESULT
)
result_directory.mkdir(parents=True, exist_ok=False)
allocated_cc_metadata = {
    "schema_version": "pytoce_allocated_cc_result_schema_v1",
    "allocated_cc_result": ALLOCATED_CC_RESULT,
    "description": "Allocated carrying capacities calculated with pyaesa.",
    "ssp_scenario": "SSP2",
    "lcia_method": ["ef31"],
    "annual_allocated_cc": "annual_allocated_cc.csv",
    "cumulative_gwp100_allocated_cc": "cumulative_gwp100_allocated_cc.csv",
}
(result_directory / "allocated_cc_result.yaml").write_text(
    yaml.safe_dump(allocated_cc_metadata, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)
annual.to_csv(
    result_directory / "annual_allocated_cc.csv",
    index=False,
    float_format="%.17g",
)
cumulative.to_csv(
    result_directory / "cumulative_gwp100_allocated_cc.csv",
    index=False,
    float_format="%.17g",
)
result_directory